# Session 3 · Exercises

*Author: course team · Licence: CC BY 4.0*

Work in pairs. The lecturer chooses which exercises to set in each block; *Stretch* tasks are for pairs that finish
early.

- **Paper** and **Predict** tasks: write your answer in the markdown cell before you run anything.
- **Code** tasks: write your query or code in the empty cells. Run the starter cell first: it opens the course
  database `case-study/data/airbnb.sqlite` (and builds it if it is missing) and defines the helper `q(sql, params)`,
  which returns a query result as a pandas DataFrame.

The database has three tables: `listings` (one row per listing, primary key `id`), `calendar` (`listing_id`,
`date`, `available` 1/0, one row per listing and night) and `reviews` (`listing_id`, `month`, `n_reviews`). Dates are
text in the form `'YYYY-MM-DD'`. You can also open the file in DB Browser for SQLite.

In [ ]:
# Starter cell: open the course database (builds it once from the case-study Parquet files if needed)
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DB = ROOT / "case-study" / "data" / "airbnb.sqlite"
if not DB.exists():
    subprocess.run([sys.executable, ROOT / "sessions/03-sql/workbooks/make_airbnb_sqlite.py"], check=True)

con = sqlite3.connect(DB)


def q(sql, params=()):
    """Run a query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params=params)


q("SELECT name FROM sqlite_master WHERE type = 'table'")

## Part 1 · Relational databases and first queries

### 1.1 · Warm-ups: predict the output

**Format:** Predict · 10 min together

> **Hand-out.** What does the database return?
> 1. `SELECT 1/2;`
> 2. `SELECT NULL = NULL;`
> 3. `SELECT 'Berlin' || NULL;`
> 4. `SELECT COUNT(*), COUNT(price), AVG(price) FROM listings;` (a third of the prices are missing)

Write your four predictions first, then check them with `q(...)`.

*Source:* [SQLite: Quirks, caveats and gotchas](https://www.sqlite.org/quirks.html); PostgreSQL docs: [mathematical operators](https://www.postgresql.org/docs/current/functions-math.html), [comparison operators](https://www.postgresql.org/docs/current/functions-comparison.html)

*Your answer:*

### 1.2 · Fermi estimate

**Format:** Paper · 5 min

> **Hand-out.** Without looking at the data: how many rows does the availability calendar have?

*Source:* [Fermi problem, Wikipedia](https://en.wikipedia.org/wiki/Fermi_problem)

*Your answer:*

### 1.3 · Count the rows of each join

**Format:** Paper · 10 min

> **Hand-out.** Table `a` has the rows (1, x), (2, y), (2, z); table `b` has (2, p), (2, q), (3, r). The first column
> is the key. How many rows do `a INNER JOIN b`, `a LEFT JOIN b` and `a CROSS JOIN b` return?

*Source:* original exercise for this course; background: [PostgreSQL: Joined tables](https://www.postgresql.org/docs/current/queries-table-expressions.html)

*Your answer:*

### 1.4 · Why can't `WHERE` use a `SELECT` alias?

**Format:** Paper · 5 min

> **Hand-out.** This query fails in PostgreSQL. Why?
> ```sql
> SELECT id, price / accommodates AS price_per_guest
> FROM listings
> WHERE price_per_guest > 50;
> ```

Then run it on the SQLite file. What do you notice?

*Source:* [PostgreSQL: SELECT, general processing](https://www.postgresql.org/docs/current/sql-select.html)

*Your answer:*

### 1.5 · First queries on the listings

**Format:** Code · 10 min

> **Hand-out.** Answer each question with one query.
> 1. How many listings require a minimum stay of 28 nights or more?
> 2. A student looks for a cheap room in Pankow: a private room under €50 a night, a minimum stay below 28 nights,
>    and at least 10 reviews. List these listings with id, neighbourhood, price, rating and number of reviews, best
>    rated first. Does any price look suspicious?
> 3. How many listings in Mitte have never been reviewed according to `first_review IS NULL`? Is that the same
>    number as with `number_of_reviews = 0`?

*Source:* original exercise for this course (Airbnb case study)

### 1.6 · Create and fill a table

**Format:** Code · 10 min

> **Hand-out.** In a new in-memory database (`sqlite3.connect(":memory:")`), create a table `trips` for your own
> last journeys with the columns `id` (primary key), `city` (text, required), `start_date` (text `'YYYY-MM-DD'`),
> `nights` (integer, must be at least 1) and `price_per_night` (decimal, may be unknown). Insert four trips, one of
> them with an unknown price. Then try to insert a trip with an `id` that exists, one with 0 nights and one without
> a city, and read the error messages. Finally, list the trips longer than three nights, the most recent first.
> What is the total cost of all trips, and how does the trip with an unknown price affect it?

*Source:* original exercise for this course; background: [SQLite: CREATE TABLE](https://www.sqlite.org/lang_createtable.html)

### 1.7 · Airbnb detective

**Format:** Code · 30 min · in teams

> **Hand-out.** Follow the clues; each answer is needed for the next query. Submit the final answer on paper.
> 1. Which host has the most listings in Berlin?
> 2. In which district does that host have the most listings?
> 3. In that district, which listing (of any host) has the most reviews?
> 4. In which month did that listing receive the most reviews?

*Note:* clues 1 and 2 count rows per host or per district. This needs `COUNT(*)` with `GROUP BY`, which block 2
teaches in full; the pattern is `SELECT host_id, COUNT(*) AS n FROM listings GROUP BY host_id ORDER BY n DESC`.

*Source:* original exercise for this course, modelled on the [SQL Murder Mystery, Knight Lab](https://github.com/NUKnightLab/sql-mysteries)

### 1.8 · How does the database find a row? (index and query plan) · *Stretch*

**Format:** Predict, then SQL · 15 min

> **Hand-out.** We look up the calendar of one listing:
> `SELECT * FROM calendar WHERE listing_id = 3176;`
> Predict: how many of the 4.7 million rows does the database read? Then run `EXPLAIN QUERY PLAN` before and after
> `CREATE INDEX idx_calendar_listing ON calendar(listing_id);` and time both queries.

When you are done, remove the index again with `DROP INDEX idx_calendar_listing;` so that the file is as before.
(Hint for timing: `import time` and `time.perf_counter()`.)

*Source:* [SQLite: EXPLAIN QUERY PLAN](https://www.sqlite.org/eqp.html)

*Your answer:*

## Part 2 · Aggregation, joins and SQL from Python

### 2.1 · Warm-up: `NOT IN` and NULL

**Format:** Predict · 5 min

> **Hand-out.** What does the database return?
> `SELECT id FROM l WHERE id NOT IN (SELECT listing_id FROM r);` when `l` has the ids 1, 2, 3 and one `listing_id`
> in `r` is NULL (`r` has the rows 1 and NULL).

Predict first, then build the two tiny tables (for example with `CREATE TEMP TABLE` in the course connection) and
check. How do you get the ids without a partner correctly?

*Source:* [PostgreSQL: Subquery expressions](https://www.postgresql.org/docs/current/functions-subquery.html); [PostgreSQL wiki: Don't Do This](https://wiki.postgresql.org/wiki/Don%27t_Do_This)

*Your answer:*

### 2.2 · Top three per district

**Format:** SQL · 15 min

> **Hand-out.** List the three most expensive listings of each district.

*Source:* [SQLite: Window functions](https://www.sqlite.org/windowfunctions.html); [PostgreSQL: Window functions](https://www.postgresql.org/docs/current/functions-window.html)

### 2.3 · Listings without a review in 2025

**Format:** SQL · 10 min

> **Hand-out.** For each district: how many listings received no review in 2025 (they may have older reviews), and
> what share of the district's listings is that? Which district has the highest share? Write it with a left join;
> where must the condition on the month go? Check one district by counting in a second way.

*Source:* original exercise for this course (Airbnb case study)

### 2.4 · The busiest night in Berlin

**Format:** Predict, then SQL · 15 min

> **Hand-out.** Which night of the year is Berlin's Airbnb market busiest? Write your guess (New Year's Eve, the
> Berlinale, the marathon, …), then find the date with the lowest share of available listings in `calendar`.

*Source:* original exercise for this course; background: [Inside Airbnb: Get the data](https://insideairbnb.com/get-the-data/)

*Your answer:*

### 2.5 · When did Berlin pass 100,000 reviews?

**Format:** SQL · 15 min

> **Hand-out.** Using the monthly review counts, find the first month in which the running total of reviews exceeded
> 100,000.

Would you report this date in a newspaper?

*Source:* original exercise for this course; background: [PostgreSQL tutorial: Window functions](https://www.postgresql.org/docs/current/tutorial-window.html)

### 2.6 · The COVID cliff

**Format:** SQL · 10 min

> **Hand-out.** With `LAG`, compare every month's number of reviews with the same month one year earlier. Which month
> had the largest drop?

*Source:* original exercise for this course; background: [SQLite: Window functions (`lag`)](https://www.sqlite.org/windowfunctions.html)

### 2.7 · Are hosts concentrated?

**Format:** SQL · 15 min

> **Hand-out.** What share of all listings belongs to the 10 % of hosts with the most listings? Guess first.

*Source:* original exercise for this course; background: [SQLite: Window functions (`ntile`)](https://www.sqlite.org/windowfunctions.html)

*Your answer:*

### 2.8 · A safe query function

**Format:** Code · 10 min

> **Hand-out.** Write a Python function `room_types(con, district, max_price)` that returns, as a pandas DataFrame,
> the number of listings and the average rating per room type for the listings of one district with a price of at
> most `max_price`. Pass both values as parameters (`?`), never with an f-string. Call it for Neukölln and €80, and
> for Mitte and €120. Then call it with the district `"Mitte' OR '1'='1"`: what happens, and what would an f-string
> version have returned?

*Source:* original exercise for this course; background: [Python docs: sqlite3, how to use placeholders](https://docs.python.org/3/library/sqlite3.html#sqlite3-placeholders)

### 2.9 · A script that loads data and answers a question

**Format:** Pairs · 20 min

> **Hand-out.** Write a script `load_weather.py` (save it in your own folder, not in the course folder) that
> 1. reads the daily Berlin weather from `case-study/data/airbnb/weather_daily.parquet`,
> 2. stores it as a table `weather` in the course database, with the date as text `'YYYY-MM-DD'` and the date as
>    primary key, so that a second run does not create duplicates,
> 3. checks the number of rows against the file, and
> 4. prints the answer to: *in which month was the mean daily temperature the highest, and how many reviews did
>    Berlin's listings receive in that month?*
>
> Run it twice from a terminal. Develop the steps in the cells below first if that helps.

*Source:* original exercise for this course; weather data: Open-Meteo historical weather API (CC BY 4.0), prepared
by `case-study/prepare_airbnb.py`

### 2.10 · The median, by hand · *Stretch*

**Format:** SQL · 15 min

> **Hand-out.** SQLite has no median function. Compute the median price of all listings anyway, and compare it with
> the mean.

*Source:* original exercise for this course; background: [PostgreSQL: Aggregate functions (`percentile_cont`)](https://www.postgresql.org/docs/current/functions-aggregate.html)

### 2.11 · Longest booked streak (gaps and islands) · *Stretch*

**Format:** SQL · 20 min

> **Hand-out.** For each listing, find the longest run of consecutive unavailable days. How many listings have a run
> of the full 365 days?

*Source:* [Camps, "The SQL of gaps and islands in sequences", Redgate Simple Talk](https://www.red-gate.com/simple-talk/databases/sql-server/t-sql-programming-sql-server/the-sql-of-gaps-and-islands-in-sequences/)

### 2.12 · Collatz in SQL · *Stretch*

**Format:** SQL · stretch

> **Hand-out.** Generate the Collatz sequence of 27 (Session 1) with a recursive CTE. Check that it needs 111 steps
> and peaks at 9,232.

*Source:* original exercise for this course; background: [SQLite: The WITH clause (recursive CTEs)](https://www.sqlite.org/lang_with.html)